# Глава 8: Создание токенизатора

> «Язык — это одежда мысли.» — **Сэмюэл Джонсон**, писатель

---

## Что такое токенизация?

**Токенизация (tokenization)** — это процесс преобразования текста в числа, чтобы нейронные сети могли его обрабатывать. Представьте это как перевод текста в секретный код, где каждое слово, подслово или символ получает уникальный номер.

```
"Hello world" → [15496, 995]  (используя токенизатор GPT-2)
```

Токенизатор также работает в обратном направлении: получая числа, он производит текст.

---

## Что вы изучите

- Как текст превращается в числа с помощью трёх различных стратегий токенизации
- Почему токенизация на уровне символов проста, но неэффективна
- Как токенизация на уровне слов обрабатывает неизвестные слова и почему важен размер словаря
- Умный приём, лежащий в основе токенизации подслов (BPE), который используется в современных LLM
- Как использовать промышленные токенизаторы, такие как tiktoken и Hugging Face transformers
- Особенности и тонкости, которые влияют на промпты и стоимость API

---

## Установка

Сначала установим необходимые пакеты:

In [ ]:
# Установка необходимых пакетов
!pip install -q tiktoken==0.14.0 transformers==5.23.0 torch==2.15.0

## 1. Токенизация на уровне символов

Давайте создадим простейший токенизатор: каждый символ — это токен.

**Напоминание о классах в Python:**
- **Класс (class)** — это шаблон, объединяющий данные и функции
- `__init__(self)` запускается при создании объекта (инициализирует его данные)
- `self` означает «этот конкретный объект» (как «эта машина» против «машины вообще»)
- `@property` заставляет метод вести себя как атрибут (скобки не нужны)

In [ ]:
class CharTokenizer:
    def __init__(self):
        # Два словаря для двунаправленного поиска
        self.char_to_id = {}
        self.id_to_char = {}

    def fit(self, text):
        """Создать словарь из текста.
        
        Зачем sorted()? Чтобы словарь был детерминированным — один и тот же текст
        всегда производит одинаковые ID. Без сортировки порядок множества в Python случаен.
        """
        # Извлечь уникальные символы, множества автоматически обрабатывают уникальность
        chars = sorted(set(text))
        
        # Присвоить каждому символу ID (начиная с 0)
        for i, c in enumerate(chars):
            self.char_to_id[c] = i
            self.id_to_char[i] = c

    def encode(self, text):
        """Преобразовать текст в список ID токенов.
        
        Возвращает список целых чисел, по одному на символ.
        """
        return [self.char_to_id[c] for c in text]

    def decode(self, ids):
        """Преобразовать список ID токенов обратно в текст.
        
        Использует str.join() для объединения символов без пробелов
        между ними (в отличие от токенизации слов, где нужны пробелы).
        """
        return "".join(self.id_to_char[i] for i in ids)

    @property
    def vocab_size(self):
        """Сколько уникальных символов мы знаем."""
        return len(self.char_to_id)

### Попробуйте: полный пример

In [ ]:
# Создать и обучить токенизатор
tokenizer = CharTokenizer()
tokenizer.fit("Hello, world!")

print(f"Размер словаря: {tokenizer.vocab_size}")
print(f"Словарь: {tokenizer.char_to_id}")

# Кодировать
text = "Hello"
ids = tokenizer.encode(text)
print(f"\n'{text}' → {ids}")

# Декодировать, обратное преобразование должно быть без потерь!
decoded = tokenizer.decode(ids)
print(f"{ids} → '{decoded}'")
print(f"Идеальное обратное преобразование? {text == decoded}")

### Компромисс между размером словаря и длиной последовательности

In [ ]:
sentence = "Tokenization is the first step in any language model."

# Уровень символов
char_tok = CharTokenizer()
char_tok.fit(sentence)
char_ids = char_tok.encode(sentence)

print(f"Исходный текст: {len(sentence)} символов")
print(f"Размер словаря: {char_tok.vocab_size}")
print(f"Длина последовательности: {len(char_ids)}")
print(f"Первые 20 токенов: {char_ids[:20]}")

## 2. Токенизация на уровне слов

Теперь создадим токенизатор на уровне слов, который обрабатывает неизвестные слова с помощью специальных токенов.

**Специальные токены (special tokens):** Зарезервированные ID с определённым значением:
- `<PAD>` (ID 0): Заполнение (padding) — заполняет последовательности до одинаковой длины для пакетной обработки
- `<UNK>` (ID 1): Неизвестный (unknown) — представляет слова, отсутствующие в нашем словаре
- `<BOS>` (ID 2): Начало последовательности (beginning of sequence) — отмечает, где начинается текст
- `<EOS>` (ID 3): Конец последовательности (end of sequence) — отмечает, где заканчивается текст

Зачем они нужны? Без `<UNK>` наш токенизатор прерывался бы на новых словах. Без `<PAD>` мы не могли бы обрабатывать несколько предложений одновременно (они имели бы разную длину).

In [ ]:
from collections import Counter

class WordTokenizer:
    def __init__(self, max_vocab_size=10000):
        """
        max_vocab_size: Максимальный размер словаря (включая специальные токены)
        
        Почему 10 000? Это компромисс:
        - Слишком маленький (1 000): Слишком много неизвестных
        - Слишком большой (100 000): Огромная таблица эмбеддингов, медленное обучение
        - 10 000-50 000: Золотая середина для обучения
        """
        self.max_vocab_size = max_vocab_size
        self.word_to_id = {}
        self.id_to_word = {}

    def fit(self, text):
        """Создать словарь из наиболее частых слов."""
        # Начнём с простого: разделение по пробелам и приведение к нижнему регистру
        words = text.lower().split()
        
        # Подсчитать частоты слов. Зачем? Распространённые слова получают свои ID,
        # редкие слова становятся <UNK>. Это минимизирует количество неизвестных на практике.
        counts = Counter(words)
        
        # Зарезервировать ID 0-3 для специальных токенов
        self.word_to_id = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<BOS>": 2,
            "<EOS>": 3
        }
        self.id_to_word = {v: k for k, v in self.word_to_id.items()}
        
        # Добавить наиболее распространённые слова (соблюдая ограничение max_vocab_size)
        # Начинать ID с 4, так как 0-3 зарезервированы для специальных токенов
        for i, (word, count) in enumerate(counts.most_common(self.max_vocab_size - 4), start=4):
            self.word_to_id[word] = i
            self.id_to_word[i] = word

    def encode(self, text, add_special_tokens=False):
        """Преобразовать текст в ID токенов.
        
        add_special_tokens: Если True, добавить <BOS> в начало и <EOS> в конец
        """
        words = text.lower().split()
        
        # Искать каждое слово, возвращаться к <UNK> (ID=1), если не найдено
        # .get(word, 1) возвращает 1, если слова нет в нашем словаре
        ids = [self.word_to_id.get(w, 1) for w in words]
        
        if add_special_tokens:
            ids = [2] + ids + [3]  # [<BOS>] + текст + [<EOS>]
        
        return ids

    def decode(self, ids, skip_special_tokens=True):
        """Преобразовать ID токенов обратно в текст.
        
        skip_special_tokens: Если True, не выводить <PAD>, <BOS> и т.д.
        Зачем? Вы не хотите вывод вида: "<BOS> Hello world <EOS>"
        """
        words = []
        for i in ids:
            word = self.id_to_word.get(i, "<UNK>")
            # Пропустить специальные токены в выводе, если запрошено
            if skip_special_tokens and word in ["<PAD>", "<BOS>", "<EOS>"]:
                continue
            words.append(word)
        
        # Объединить с пробелами (в отличие от токенизатора символов, который использовал "".join)
        return " ".join(words)

    @property
    def vocab_size(self):
        """Текущий размер словаря (количество уникальных токенов)."""
        return len(self.word_to_id)

### Попробуйте: полный пример с неизвестными словами

In [ ]:
# Создать токенизатор с небольшим словарём, чтобы появились неизвестные слова
tokenizer = WordTokenizer(max_vocab_size=10)

# Обучить на ограниченном тексте
training_text = """
The cat sat on the mat.
The cat was on the mat.
The dog sat on the mat.
"""
tokenizer.fit(training_text)

print(f"Словарь: {tokenizer.word_to_id}")

# Закодировать предложение с известными словами
text1 = "the cat sat"
ids1 = tokenizer.encode(text1)
print(f"\n'{text1}' → {ids1}")
print(f"Декодировано: '{tokenizer.decode(ids1)}'")

# Закодировать с неизвестным словом
text2 = "the elephant sat"  # "elephant" нет в словаре!
ids2 = tokenizer.encode(text2)
print(f"\n'{text2}' → {ids2}")
print(f"Декодировано: '{tokenizer.decode(ids2)}'")

# Попробовать со специальными токенами
ids3 = tokenizer.encode("the cat", add_special_tokens=True)
print(f"\nСо специальными токенами: {ids3}")
print(f"Декодировано (показаны специальные): '{tokenizer.decode(ids3, skip_special_tokens=False)}'")
print(f"Декодировано (скрыты специальные): '{tokenizer.decode(ids3, skip_special_tokens=True)}'")

### Упражнение для сравнения: увидьте компромисс

In [ ]:
text = "The quick brown fox jumps over the lazy dog"

# Токенизатор символов
char_tok = CharTokenizer()
char_tok.fit(text)
char_ids = char_tok.encode(text)

# Токенизатор слов
word_tok = WordTokenizer(max_vocab_size=20)
word_tok.fit(text)
word_ids = word_tok.encode(text)

print("ТОКЕНИЗАТОР СИМВОЛОВ:")
print(f"  Размер словаря: {char_tok.vocab_size}")
print(f"  Длина последовательности: {len(char_ids)}")
print(f"  Токены: {char_ids[:20]}...")

print("\nТОКЕНИЗАТОР СЛОВ:")
print(f"  Размер словаря: {word_tok.vocab_size}")
print(f"  Длина последовательности: {len(word_ids)}")
print(f"  Токены: {word_ids}")

## Как работают промышленные токенизаторы: BPE (кодирование парами байтов)

Современные LLM используют **токенизацию подслов (subword tokenization)** — умную золотую середину между символами и словами:

**Проблема:**
- Токенизаторы символов: слишком много токенов на текст (медленно, дорого)
- Токенизаторы слов: не могут обрабатывать новые слова ("ChatGPT" → `<UNK>`)

**Решение: BPE (Byte-Pair Encoding, кодирование парами байтов)**

BPE автоматически изучает подслова, повторно объединяя наиболее частые пары символов:

```
Шаг 1: Начать с символов: ["l", "o", "w", "e", "r"]
Шаг 2: Наиболее частая пара ("l", "o") → объединить в "lo"
Шаг 3: Наиболее частая пара ("lo", "w") → объединить в "low"
Шаг 4: Продолжать, пока не достигнут размер словаря...
```

**Результат:** Распространённые слова становятся отдельными токенами, редкие слова разделяются на известные части:
- "lower" → ["low", "er"] ✓ (распространённое, эффективно)
- "lowest" → ["low", "est"] ✓ (составное слово обработано!)
- "ChatGPT" → ["Chat", "G", "PT"] ✓ (не нужен `<UNK>`!)

**Ключевая идея:** BPE никогда не производит `<UNK>`, потому что любая последовательность символов может быть разбита на известные части!

---

## 3. Промышленные токенизаторы: tiktoken

Теперь давайте используем библиотеку tiktoken от OpenAI с кодировкой `cl100k_base` — кодировкой эпохи GPT-3.5/GPT-4 (2023). Современные модели OpenAI используют более крупную `o200k_base`; механика идентична.

## Обучение собственного BPE-токенизатора

Теперь давайте увидим BPE в действии! Мы обучим собственный токенизатор на текстах Шекспира, используя библиотеку HuggingFace `tokenizers`.

**Зачем это делать?**
- Увидеть, как паттерны возникают автоматически ("ing", "the", "tion")
- Развить интуицию о том, как работают токенизаторы
- Понять несоответствие областей (токенизатор Шекспира против современного текста)

Сначала установим библиотеку tokenizers:

In [ ]:
# Установить библиотеку tokenizers (отдельно от transformers!)
!pip install -q tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print("Импорты успешны!")

### Загрузить TinyShakespeare

Мы используем тот же набор данных, на котором вы будете обучать полную LLM в главе 12:

In [ ]:
import urllib.request

# Загрузить TinyShakespeare (тот же набор данных, который мы будем использовать в главе 12!)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

# Прочитать и изучить
with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Размер набора данных: {len(text):,} символов")
print(f"\nПервые 200 символов:")
print(text[:200])

### Обучить BPE-токенизатор

Теперь самая захватывающая часть — наблюдайте, как BPE изучает паттерны из Шекспира!

In [ ]:
# Создать пустой BPE-токенизатор
bpe_tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

# Разделить по пробелам до объединения BPE
bpe_tokenizer.pre_tokenizer = Whitespace()

# Настроить обучающий модуль
trainer = BpeTrainer(
    vocab_size=1000,           # Маленький для обучения (в промышленности используется 30 000+)
    min_frequency=2,           # Токен должен встречаться минимум дважды
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
)

# Обучить на Шекспире (это займёт всего несколько секунд!)
bpe_tokenizer.train(files=["shakespeare.txt"], trainer=trainer)

print(f"Размер словаря: {bpe_tokenizer.get_vocab_size()}")

### Протестировать обученный токенизатор

In [ ]:
# Тест на тексте в стиле Шекспира
test_text = "To be or not to be, that is the question."
output = bpe_tokenizer.encode(test_text)

print(f"Текст: '{test_text}'")
print(f"Токены: {output.tokens}")
print(f"ID токенов: {output.ids}")
print(f"Количество токенов: {len(output.ids)}")

# Проверить обратное преобразование
decoded = bpe_tokenizer.decode(output.ids)
print(f"\nДекодировано: '{decoded}'")
print(f"Идеальное обратное преобразование: {test_text == decoded}")

### Несоответствие областей: современный текст на токенизаторе Шекспира

Что происходит, когда мы пробуем современный технический текст, который Шекспир никогда не писал?

In [ ]:
# Попробовать современный текст, который Шекспир никогда не писал
modern_text = "ChatGPT generates amazing neural network responses"
output2 = bpe_tokenizer.encode(modern_text)

print(f"Текст: '{modern_text}'")
print(f"Токены: {output2.tokens}")
print(f"Количество токенов: {len(output2.ids)}")

# Обратите внимание: "ing" — это токен (Шекспир его использовал!), но "ChatGPT" разделяется на части
print("\nКлючевая идея:")
print("- 'ing' — это один токен (Шекспир часто использовал слова с '-ing')")
print("- 'ChatGPT' разделяется на части (Шекспир не писал об ИИ!)")
print("- Это несоответствие областей в действии")

---

## Промышленные токенизаторы: tiktoken

Теперь вы видели, как BPE учится с нуля. Вы понимаете, что находится внутри этих промышленных токенизаторов. Когда tiktoken разделяет "ChatGPT" на части, вы знаете почему!

Давайте используем промышленные инструменты, которые вы реально будете использовать в своих проектах.

In [ ]:
import tiktoken

# Загрузить кодировку токенизатора эпохи GPT-3.5/GPT-4 (2023)
enc = tiktoken.get_encoding("cl100k_base")  # эпоха GPT-3.5/GPT-4; современные модели OpenAI используют o200k_base

# Закодировать текст
text = "Hello, world! How are you?"
tokens = enc.encode(text)

print(f"Текст: '{text}'")
print(f"Токены: {tokens}")
print(f"Количество токенов: {len(tokens)}")

# Декодировать обратно
decoded = enc.decode(tokens)
print(f"Декодировано: '{decoded}'")
print(f"Идеальное обратное преобразование: {text == decoded}")

### Посмотреть фактические строки токенов

In [ ]:
# Декодировать каждый токен индивидуально, чтобы увидеть, что он представляет
token_strings = [enc.decode([t]) for t in tokens]

print(f"Разбивка токенов:")
for token_id, token_str in zip(tokens, token_strings):
    print(f"  {token_id:5d} → '{token_str}'")

### Подсчёт токенов для бюджетирования API

In [ ]:
prompts = [
    "Write a haiku about programming.",
    "Explain quantum computing in simple terms for a 10-year-old child.",
    "Generate a 500-word essay on climate change."
]

enc = tiktoken.get_encoding("cl100k_base")

for prompt in prompts:
    tokens = enc.encode(prompt)
    # Пример ценообразования (проверьте текущие тарифы на openai.com/api/pricing)
    cost = len(tokens) * 0.00001  # $0.01 за 1K входных токенов
    
    print(f"Промпт: '{prompt}'")
    print(f"  Токенов: {len(tokens)}")
    print(f"  Стоимость (входных): ~${cost:.5f}\n")

## 4. Промышленные токенизаторы: Hugging Face Transformers

**Hugging Face** — это компания/библиотека, предоставляющая предобученные модели и токенизаторы. `AutoTokenizer` автоматически загружает правильный токенизатор для любой модели.

Токенизатор Hugging Face возвращает словарь с:
- `input_ids`: ID токенов (то, что нас больше всего интересует)
- `attention_mask`: единицы для реальных токенов, нули для заполнения (подсказывает модели, что игнорировать)

In [ ]:
from transformers import AutoTokenizer

# Загрузить токенизатор GPT-2 (с открытым исходным кодом)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Hello, world! How are you?"

# Закодировать, возвращает словарь с ID токенов и маской внимания
encoded = tokenizer(text, return_tensors="pt")  # "pt" = тензоры PyTorch

print(f"Текст: '{text}'")
print(f"ID токенов: {encoded['input_ids']}")
print(f"Маска внимания: {encoded['attention_mask']}")

# Декодировать
decoded = tokenizer.decode(encoded['input_ids'][0])
print(f"Декодировано: '{decoded}'")

### Разбивка токенов

In [ ]:
tokens = encoded['input_ids'][0].tolist()
token_strings = [tokenizer.decode([t]) for t in tokens]

print(f"Разбивка токенов:")
for tid, tstr in zip(tokens, token_strings):
    print(f"  {tid:5d} → '{tstr}'")

## 5. Особенности токенизации

### Особенность #1: Начальные пробелы меняют всё

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Сравнить с начальным пробелом и без него
texts = ["Hello", " Hello", "world", " world"]

for text in texts:
    tokens = enc.encode(text)
    print(f"'{text}' → {tokens} ({len(tokens)} токен{'ов' if len(tokens) > 1 else ''})")

### Особенность #2: Числа разделяются по цифрам

In [ ]:
numbers = ["10", "100", "1000", "10000", "42", "2024"]

enc = tiktoken.get_encoding("cl100k_base")

for num in numbers:
    tokens = enc.encode(num)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{num}' → {tokens} = {token_strs}")

### Особенность #3: Эмодзи и специальные символы

In [ ]:
emojis = ["😀", "🚀", "👍", "Hello 😀 world", "🔥🔥🔥"]

enc = tiktoken.get_encoding("cl100k_base")

for text in emojis:
    tokens = enc.encode(text)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{text}' → {len(tokens)} токенов: {token_strs}")

## 6. Практическое упражнение: токенизация вашего набора данных

Соединим части. Возьмите `train.jsonl` — файл, который сохранил блокнот главы 7, — и токенизируйте его токенизатором GPT-2. Почему GPT-2, а не tiktoken? Потому что главы 9-12 строят модель в стиле GPT-2 со словарём на 50 257 токенов, и сохраняемые здесь ID — это именно те, которые та модель сможет использовать. (Сначала выполните блокнот главы 7 или положите `train.jsonl` в рабочий каталог этого блокнота.)

В Colab среда каждого блокнота отдельная. В главе 7 выполните `from google.colab import files; files.download("train.jsonl")`, а здесь — `from google.colab import files; files.upload()` и выберите файл. Другой вариант — выполнить ячейки конвейера главы 7 в этой среде. Локально достаточно общей рабочей папки.


In [ ]:
import json
from transformers import GPT2TokenizerFast

# Загрузите набор данных главы 7 (сохранён блокнотом главы 7)
from pathlib import Path
if not Path("train.jsonl").is_file():
    raise FileNotFoundError("Загрузите train.jsonl из главы 7 через files.upload() или выполните её конвейер в этой среде.")
with open("train.jsonl", "r", encoding="utf-8") as f:
    dataset = [json.loads(line) for line in f]

# Токенизатор GPT-2: тот же словарь, что используют главы 9-12
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

for record in dataset:
    tokens = tokenizer.encode(record["text"])
    record["token_ids"] = tokens
    record["token_count"] = len(tokens)

# Сохраняем токенизированный набор; метки split переносятся без изменений
with open("chapter8_output.jsonl", "w", encoding="utf-8") as f:
    for record in dataset:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Токенизировано {len(dataset)} примеров")

token_counts = [r["token_count"] for r in dataset]
avg_tokens = sum(token_counts) / len(token_counts)
max_tokens = max(token_counts)
min_tokens = min(token_counts)

print(f"\nСтатистика:")
print(f"  Среднее количество токенов на пример: {avg_tokens:.1f}")
print(f"  Максимум токенов: {max_tokens}")
print(f"  Минимум токенов: {min_tokens}")

# Быстрое сравнение: то же предложение через API-токенизатор даёт другие ID
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
text = dataset[0]["text"]
print(f"ID GPT-2:  {tokenizer.encode(text)}")
print(f"ID cl100k: {enc.encode(text)}")

# Разные словари означают разные ID для одного текста.
# ID cl100k — это то, за что выставляют счёт API OpenAI; ID GPT-2 —
# то, что понимает ваш MiniGPT, поэтому мы сохранили именно их.


## Резюме главы

**Что мы построили:**

1. **Токенизатор символов:** Простой, но неэффективный (маленький словарь ~100, длинные последовательности)
2. **Токенизатор слов:** Эффективные последовательности, но огромный словарь и проблемы с неизвестными словами
3. **BPE-токенизатор:** Обучили свой собственный на Шекспире, увидели автоматическое появление паттернов!
4. **Промышленные инструменты:** Использовали tiktoken и Hugging Face для реальной токенизации

**Что мы узнали:**

- Токенизация обратима (обратное преобразование без потерь)
- Компромисс между размером словаря и длиной последовательности является фундаментальным
- Специальные токены служат конкретным целям (BOS/EOS/PAD/UNK)
- BPE изучает подслова автоматически, объединяя частые пары
- Несоответствие областей имеет значение: токенизатор Шекспира плохо справляется с современными техническими словами
- Токенизация имеет особенности (начальные пробелы, разделение чисел, эмодзи)

**Далее:** Глава 9 преобразует эти ID токенов в векторы эмбеддингов!